In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
from scipy.stats import chi2_contingency, fisher_exact

sc.set_figure_params(dpi=100, color_map='viridis_r', transparent=False, frameon=False)
sc.settings.verbosity = 1
sc.logging.print_header()

FIG_DIR = './figures/6_Combine_subclusters'
import os; os.makedirs(FIG_DIR, exist_ok=True)
sc.settings.figdir = FIG_DIR
%config InlineBackend.figure_format = 'retina'

import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

## 加载数据

In [ ]:
adata = sc.read_h5ad('adata2.h5ad')
adata.uns['log1p']['base'] = None
print(adata.shape)
adata.obs[['cell_type', 'tissue_type', 'Minor_cluster']].head()

## 补全未注释细胞

Sub-clustering 仅覆盖了 4 个 compartment (Myeloid/T-NK/Tumor/Stromal)，
剩余 ~10k 未注释细胞 (Prol.tumor, OPC-like Tumor, NPC-like Tumor) 用原始 cell_type 填充。

In [ ]:
na_mask = adata.obs['Minor_cluster'].isna()
print(f'未注释: {na_mask.sum()} cells')
print(adata.obs.loc[na_mask, 'cell_type'].value_counts())

# 转 string 后填充 (避免 Categorical 冲突)
minor = adata.obs['Minor_cluster'].astype(str)
minor[minor == 'nan'] = adata.obs.loc[minor == 'nan', 'cell_type'].astype(str)
adata.obs['Minor_cluster'] = pd.Categorical(minor)
print(f'\n填充后共 {adata.obs["Minor_cluster"].nunique()} 个亚型')

## Major category 分组

In [ ]:
major_map = {
    # Myeloid / TAM
    'Mg-Inflam TAM': 'TAM/Macrophage',
    'Complement-Scavenger TAM': 'TAM/Macrophage',
    'Lipid-metabolism TAM': 'TAM/Macrophage',
    'Systemic-Inflam TAM': 'TAM/Macrophage',
    'Other TAM': 'TAM/Macrophage',
    'HS-UPR TAM': 'TAM/Macrophage',
    'IFN-resp TAM': 'TAM/Macrophage',
    # Microglia
    'Homeostatic MG': 'Microglia',
    # Monocyte
    'Classical Mono': 'Monocyte',
    'Neutrophil': 'Neutrophil',
    # T/NK
    'CD8 effector': 'T/NK',
    'CD4 Th': 'T/NK',
    'CD4 memory': 'T/NK',
    'γδ T': 'T/NK',
    'CD8 pre-exhausted': 'T/NK',
    'NK': 'T/NK',
    'Cycling T': 'T/NK',
    'T-Tumor doublet': 'Doublet',
    'Low-quality T': 'Low-quality',
    # Tumor
    'AC': 'Tumor',
    'OPC': 'Tumor',
    'MES1': 'Tumor',
    'NPC2': 'Tumor',
    'NPC1': 'Tumor',
    'Prol.tumor': 'Tumor',
    'OPC-like Tumor': 'Tumor',
    'NPC-like Tumor': 'Tumor',
    # Stromal
    'Activated EC': 'Endothelial',
    'Venous EC': 'Endothelial',
    'Arterial EC': 'Endothelial',
    'Tip cell': 'Endothelial',
    'BBB-like EC': 'Endothelial',
    'Mural cell': 'Pericyte',
    'CNS Pericyte': 'Pericyte',
    'Classic Pericyte': 'Pericyte',
    'Doublet (Ig+)': 'Doublet',
}

adata.obs['Major_category'] = adata.obs['Minor_cluster'].map(major_map).astype('category')
print(adata.obs['Major_category'].value_counts())

## 全景 UMAP

In [ ]:
sc.set_figure_params(dpi=150, color_map='viridis_r', fontsize=6, transparent=False, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(24, 7))
sc.pl.umap(adata, color='Major_category', ax=axes[0], show=False,
           title='Major category', legend_loc='right margin')
sc.pl.umap(adata, color='tissue_type', ax=axes[1], show=False,
           title='Tissue type', legend_loc='right margin')
sc.pl.umap(adata, color='sample', ax=axes[2], show=False,
           title='Sample', legend_loc='right margin', legend_fontsize=5)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/umap_overview.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
sc.pl.umap(adata, color='Minor_cluster', legend_loc='right margin',
           legend_fontsize=5, title='All subtypes (Minor_cluster)',
           save='_all_subtypes.png')

## 瘤旁 vs 肿瘤 UMAP 对比

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, region in zip(axes, ['Peritumoral', 'Tumor']):
    mask = adata.obs['tissue_type'] == region
    sc.pl.umap(adata[mask], color='Minor_cluster', ax=ax, show=False,
               title=f'{region} ({mask.sum():,} cells)', legend_loc='none')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/umap_region_split.png', dpi=200, bbox_inches='tight')
plt.show()

## 细胞组成分析

In [ ]:
# Major category 比例: 瘤旁 vs 肿瘤
ct = pd.crosstab(adata.obs['tissue_type'], adata.obs['Major_category'], normalize='index')

fig, ax = plt.subplots(figsize=(8, 5))
ct.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_ylabel('Proportion')
ax.set_title('Major category composition by region')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/composition_major_by_region.png', dpi=200, bbox_inches='tight')
plt.show()

print(ct.round(3).to_string())

In [ ]:
# Minor_cluster 比例 (top 20)
ct_minor = pd.crosstab(adata.obs['tissue_type'], adata.obs['Minor_cluster'], normalize='index')
top_clusters = ct_minor.mean().nlargest(20).index

fig, ax = plt.subplots(figsize=(12, 5))
ct_minor[top_clusters].plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_ylabel('Proportion')
ax.set_title('Top 20 Minor_cluster composition by region')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/composition_minor_by_region.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# 每个样本的 Major category 比例
ct_sample = pd.crosstab(adata.obs['sample'], adata.obs['Major_category'], normalize='index')
ct_sample['tissue_type'] = adata.obs.groupby('sample')['tissue_type'].first()
ct_sample = ct_sample.sort_values('tissue_type')

fig, ax = plt.subplots(figsize=(12, 5))
ct_sample.drop(columns='tissue_type').plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_ylabel('Proportion')
ax.set_title('Major category per sample')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/composition_per_sample.png', dpi=200, bbox_inches='tight')
plt.show()

## 区域差异统计检验

In [ ]:
# 逐亚型 chi-square (Major_category)
ct_abs = pd.crosstab(adata.obs['tissue_type'], adata.obs['Major_category'])
chi2, pval, dof, expected = chi2_contingency(ct_abs)
print(f'Overall chi-square: chi2={chi2:.1f}, p={pval:.2e}, dof={dof}')

# 逐亚型: 哪些在肿瘤/瘤旁富集
results = []
total_per_region = adata.obs['tissue_type'].value_counts()
for cat in ct_abs.columns:
    n_peri = ct_abs.loc['Peritumoral', cat]
    n_tumor = ct_abs.loc['Tumor', cat]
    # proportion
    prop_peri = n_peri / total_per_region['Peritumoral']
    prop_tumor = n_tumor / total_per_region['Tumor']
    fold = prop_tumor / prop_peri if prop_peri > 0 else np.inf
    # 2x2 table for this category vs rest
    table_2x2 = np.array([[n_tumor, total_per_region['Tumor'] - n_tumor],
                          [n_peri, total_per_region['Peritumoral'] - n_peri]])
    chi2_i, p_i, _, _ = chi2_contingency(table_2x2)
    enriched = 'Tumor' if fold > 1 else 'Peritumoral'
    results.append({'category': cat, 'n_Peri': n_peri, 'n_Tumor': n_tumor,
                    'prop_Peri': prop_peri, 'prop_Tumor': prop_tumor,
                    'fold_T/P': fold, 'enriched_in': enriched,
                    'chi2': chi2_i, 'pval': p_i})

df_stat = pd.DataFrame(results).sort_values('pval')
df_stat['pval_str'] = df_stat['pval'].apply(lambda x: f'{x:.2e}')
print(df_stat[['category', 'n_Peri', 'n_Tumor', 'prop_Peri', 'prop_Tumor',
               'fold_T/P', 'enriched_in', 'pval_str']].to_string(index=False))

In [ ]:
# 富集方向可视化 (log2 fold change)
df_stat['log2FC'] = np.log2(df_stat['fold_T/P'].replace([np.inf, 0], np.nan))
df_plot = df_stat.dropna(subset=['log2FC']).sort_values('log2FC')

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2166ac' if x < 0 else '#b2182b' for x in df_plot['log2FC']]
ax.barh(df_plot['category'], df_plot['log2FC'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('log2(Tumor / Peritumoral proportion)')
ax.set_title('Cell type enrichment: Tumor vs Peritumoral')
# 标注 p-value
for i, row in df_plot.iterrows():
    sig = '***' if row['pval'] < 0.001 else '**' if row['pval'] < 0.01 else '*' if row['pval'] < 0.05 else 'ns'
    offset = 0.05 if row['log2FC'] >= 0 else -0.05
    ax.text(row['log2FC'] + offset, row['category'], sig, va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/enrichment_barplot.png', dpi=200, bbox_inches='tight')
plt.show()

## 通路评分: 瘤旁 vs 肿瘤

In [ ]:
# 关键 GBM 通路基因集
pathway_genes = {
    'Hypoxia': ['HIF1A', 'VEGFA', 'SLC2A1', 'LDHA', 'PGK1', 'ENO1', 'ALDOA', 'CA9', 'BNIP3'],
    'IFN_response': ['ISG15', 'IFI6', 'IFIT1', 'IFIT3', 'IFITM1', 'IFITM3', 'MX1', 'OAS1', 'STAT1'],
    'EMT': ['VIM', 'CDH2', 'SNAI1', 'SNAI2', 'ZEB1', 'ZEB2', 'TWIST1', 'FN1', 'MMP2'],
    'Glycolysis': ['SLC2A1', 'HK2', 'PFKP', 'PKM', 'LDHA', 'ENO1', 'GAPDH', 'PGK1', 'ALDOA'],
    'Antigen_presentation': ['HLA-A', 'HLA-B', 'HLA-C', 'B2M', 'TAP1', 'TAP2', 'PSMB8', 'PSMB9'],
    'Inflammasome': ['NLRP3', 'CASP1', 'IL1B', 'IL18', 'GSDMD', 'PYCARD', 'NFKB1'],
}

existing = set(adata.var_names)
for name, genes in pathway_genes.items():
    genes_use = [g for g in genes if g in existing]
    if len(genes_use) >= 3:
        sc.tl.score_genes(adata, genes_use, score_name=f'{name}_score')
        print(f'{name}: {len(genes_use)}/{len(genes)} genes used')
    else:
        print(f'{name}: SKIP (only {len(genes_use)} genes found)')

In [ ]:
# 通路得分: 按 Major_category × tissue_type 可视化
score_cols = [c for c in adata.obs.columns if c.endswith('_score') and c.startswith(tuple(pathway_genes.keys()))]

df_scores = sc.get.obs_df(adata, score_cols + ['Major_category', 'tissue_type'])
# 去掉 Doublet 和 Low-quality
df_scores = df_scores[~df_scores['Major_category'].isin(['Doublet', 'Low-quality'])]
df_long = df_scores.melt(id_vars=['Major_category', 'tissue_type'], value_vars=score_cols,
                         var_name='pathway', value_name='score')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, pathway in zip(axes.flat, score_cols):
    sub = df_long[df_long['pathway'] == pathway]
    sns.boxplot(data=sub, x='Major_category', y='score', hue='tissue_type',
                palette={'Peritumoral': '#4393c3', 'Tumor': '#d6604d'},
                fliersize=0.5, linewidth=0.8, ax=ax)
    ax.set_title(pathway.replace('_score', ''), fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=6)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/pathway_scores_by_region.png', dpi=200, bbox_inches='tight')
plt.show()

## 关键基因表达: 瘤旁 vs 肿瘤

In [ ]:
# 免疫检查点 + 免疫抑制 + 炎症关键基因
genelist = ['CD274', 'PDCD1', 'CTLA4', 'LAG3', 'TIGIT', 'HAVCR2',  # checkpoint
            'IDO1', 'ARG1', 'CD47', 'TGFB1',  # immunosuppression
            'SPP1', 'MIF', 'CXCL12', 'CCL2', 'IL1B', 'TNF']  # inflammation
genelist_use = [g for g in genelist if g in adata.var_names]

df = sc.get.obs_df(adata, genelist_use + ['Major_category', 'tissue_type'])
df = df[~df['Major_category'].isin(['Doublet', 'Low-quality'])]
df_long = df.melt(id_vars=['Major_category', 'tissue_type'], value_vars=genelist_use)

# 选几个最关键的画 barplot
genes_show = ['CD274', 'SPP1', 'IDO1', 'TGFB1', 'CCL2', 'IL1B']
genes_show = [g for g in genes_show if g in genelist_use]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, gene in zip(axes.flat, genes_show):
    sub = df_long[df_long['variable'] == gene]
    sns.barplot(data=sub, x='Major_category', y='value', hue='tissue_type',
                hue_order=['Peritumoral', 'Tumor'],
                palette={'Peritumoral': '#4393c3', 'Tumor': '#d6604d'},
                errorbar='se', ax=ax)
    ax.set_title(gene, fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Expression')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/key_genes_by_region.png', dpi=200, bbox_inches='tight')
plt.show()

## Marker dotplot

In [ ]:
marker_dict = {
    'TAM': ['CD163', 'MRC1', 'MSR1', 'SPP1', 'C1QA'],
    'Microglia': ['P2RY12', 'TMEM119', 'CX3CR1'],
    'Monocyte': ['S100A8', 'S100A9', 'FCN1', 'VCAN'],
    'Neutrophil': ['CXCR2', 'FCGR3B', 'CSF3R'],
    'T cell': ['CD3D', 'CD8A', 'CD4', 'FOXP3', 'NKG7'],
    'Tumor': ['SOX2', 'OLIG1', 'OLIG2', 'EGFR', 'MKI67'],
    'Endothelial': ['PECAM1', 'VWF', 'CLDN5', 'FLT1'],
    'Pericyte': ['PDGFRB', 'RGS5', 'ACTA2'],
}

# 过滤不存在的基因
marker_dict_use = {k: [g for g in v if g in adata.var_names] for k, v in marker_dict.items()}

# 排除 Doublet/Low-quality
adata_clean = adata[~adata.obs['Major_category'].isin(['Doublet', 'Low-quality'])].copy()

sc.pl.dotplot(adata_clean, marker_dict_use, groupby='Major_category',
              dendrogram=False, standard_scale='var',
              save='_major_markers.png')

In [ ]:
# 按 Minor_cluster 的 matrixplot (去掉 Doublet/Low-quality)
sc.pl.matrixplot(adata_clean, marker_dict_use, groupby='Minor_cluster',
                 dendrogram=True, standard_scale='var',
                 cmap='RdBu_r', save='_minor_markers.png')

## 保存

In [ ]:
# 保存增强后的 adata (含 Major_category + pathway scores)
adata.write('adata2.h5ad')
print('adata2.h5ad 已更新 (新增 Major_category + pathway scores)')
print(f'  shape: {adata.shape}')
print(f'  Minor_cluster: {adata.obs["Minor_cluster"].nunique()} types')
print(f'  Major_category: {adata.obs["Major_category"].nunique()} groups')